<a href="https://www.kaggle.com/code/shamanthv/flight-price?scriptVersionId=351927335" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Block 1: Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
import xgboost as xgb

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings("ignore")


In [ ]:
sample_submission = pd.read_csv("/kaggle/input/mlp-term-2-2025-kaggle-assignment-1/sample_submission.csv")
train = pd.read_csv("/kaggle/input/mlp-term-2-2025-kaggle-assignment-1/train.csv")
test = pd.read_csv("/kaggle/input/mlp-term-2-2025-kaggle-assignment-1/test.csv")
train.head()

In [ ]:
print("Column data types:\n")
print(train.dtypes)

In [ ]:
print(train.describe())
print("\nMedian values:\n")
print(train.median(numeric_only=True))


In [ ]:
# Block 5: Check and handle missing values

print("Missing values in train:\n", train.isnull().sum())
print("\nMissing values in test:\n", test.isnull().sum())

# Separate column types
num_cols = train.select_dtypes(include=np.number).columns.tolist()
cat_cols = train.select_dtypes(include='object').columns.tolist()

# Remove target column ('price') from numeric columns before applying to test
if 'price' in num_cols:
    num_cols.remove('price')

# Fill numeric missing values with median
for col in num_cols:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    if col in test.columns:
        test[col] = test[col].fillna(median_val)

# Fill categorical missing values with mode
for col in cat_cols:
    mode_val = train[col].mode()[0]
    train[col] = train[col].fillna(mode_val)
    if col in test.columns:
        test[col] = test[col].fillna(mode_val)


In [ ]:
# Block 6: Check for duplicates
print("Duplicates in training data:", train.duplicated().sum())
train = train.drop_duplicates()


In [ ]:
# Block 7: Checking for outliers (visual check)
sns.boxplot(train['price'])
plt.title('Outlier check for price')
plt.show()

# Capping extreme outliers
q1 = train['price'].quantile(0.25)
q3 = train['price'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

train['price'] = np.where(train['price'] > upper, upper, train['price'])
train['price'] = np.where(train['price'] < lower, lower, train['price'])


In [ ]:
# Block 8: Visualizations

# 1. Price by airline
plt.figure(figsize=(10, 4))
sns.boxplot(x='airline', y='price', data=train)
plt.xticks(rotation=45)
plt.title("Flight Price vs Airline")
plt.show()

# 2. Price by class
sns.boxplot(x='class', y='price', data=train)
plt.title("Price vs Class")
plt.show()

# 3. Correlation heatmap
sns.heatmap(train.corr(numeric_only=True), annot=True)
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
# Block 9: Preprocessing
target = 'price'
X = train.drop(columns=['id', 'flight', target])
y = train[target]
X_test = test.drop(columns=['id', 'flight'])

numerical_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

num_transform = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transform = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_transform, numerical_cols),
    ('cat', cat_transform, categorical_cols)
])


In [ ]:
# Block 10: Train-test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=1)


In [ ]:
# Block 11: Train 10 models and compare RMSE on validation set

models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=50, random_state=1),
    'DecisionTree': DecisionTreeRegressor(random_state=1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=50, random_state=1),
    'KNN': KNeighborsRegressor(),
    'SVR': SVR(),
    'XGBoost': xgb.XGBRegressor(random_state=1, verbosity=0),
    'ExtraTrees': RandomForestRegressor(n_estimators=50, max_depth=10, random_state=1)  # using RF to simulate ExtraTrees
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    pipe = Pipeline([
        ('preprocess', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    val_preds = pipe.predict(X_val)
    rmse = mean_squared_error(y_val, val_preds, squared=False)
    results[name] = rmse
    print(name, "RMSE:", rmse)


In [ ]:
# Block 12: Final Model Comparison (simplified)
try:
    best_model = min(results, key=results.get)
    print(f"\nBest model based on validation RMSE: {best_model}")
except Exception as e:
    print("Error in comparing models:", e)


In [ ]:
# ─── Block 13: Final Prediction and Submission ───────────────────────────────

# Use the model pipeline again to predict test set
try:
    final_model = models[best_model]

    final_pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', final_model)
    ])

    # Fit on full training data
    final_pipe.fit(X, y)

    # Predict on test data
    test_preds = final_pipe.predict(test)

    # Clip predictions to remove negatives (optional)
    test_preds = np.clip(test_preds, 0, None)

    # Load sample_submission.csv safely
    submission = pd.read_csv('/kaggle/input/mlp-term-2-2025-kaggle-assignment-1/sample_submission.csv')
    submission['price'] = test_preds
    submission.to_csv('submission.csv', index=False)

    print("✅ submission.csv saved successfully.")

except Exception as e:
    print("❌ Error while predicting or saving submission:", e)


In [ ]:
from IPython.display import FileLink
FileLink("submission.csv")
